# Human-Centered AI End-to-End Workshop Notebook

This notebook provides an end-to-end, publication-grade workflow for two projects: 
1) **Project 1:** Multimodal LLM co-creation of a videogame (Gemini model orchestration).
2) **Project 2:** Human-centered workload planner using **AutoGluon** for AutoML.

Each section includes **Project 1** and **Project 2** subsections, automated analysis pipelines, and a parallel path for manual methods.

> **Note:** Pre-built outputs and dummy data are stored in `deliverables/`.


## 0) Environment Setup

Install the dependencies used in the notebook. In class, you may skip large installations and run only the relevant sections.


In [ ]:
!pip -q install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn python-dotenv google-generativeai autogluon.tabular flask


## 1) Project Definition

### Project 1: Multimodal LLM Game Co-Creation
**Human-Centered Methods:** contextual inquiry, participatory design, iterative prototyping, and summative usability testing.

**Novelty framing (CHI-ready):** A multi-model orchestration pipeline that co-creates *narrative*, *visual style*, and *music* with explicit user preference elicitation and transparency mechanisms. The study demonstrates how model outputs can be constrained by co-created human values and verified through mixed-method triangulation.

**Research Questions:**
- RQ1: How does multimodal co-creation affect perceived creative agency and trust?
- RQ2: Which multimodal elements (story, music, style) most influence perceived immersion and usability?

**Independent Variables:** co-creation mode (guided vs. exploratory), transparency level (high vs. low).
**Dependent Variables:** creative agency, trust, SUS, immersion (Likert), task success, time on task.
**Hypotheses:** H1 guided mode increases agency; H2 high transparency increases trust.

**Power analysis (example):** medium effect size (d=0.5), alpha=0.05, power=0.8 → ~64 participants per condition.

**Recruitment & demographics:** recruit from campus/community channels, balance gender identity, include accessibility needs, collect age, gaming experience, and prior AI exposure.
**Ethics:** IRB/ethics approval, informed consent, opt-out, and data minimization.

### Project 2: Human-Centered Workload Planner (AutoGluon)
**Human-Centered Methods:** survey-based requirements, explainability workshops, accessibility evaluation, and mixed-method usability testing.

**Novelty framing (CHI-ready):** A student workload planner using AutoML predictions with transparent, user-configurable explanations and human override mechanisms. Demonstrates how model transparency and user control affect decision confidence.

**Research Questions:**
- RQ1: Does explainable AutoML improve decision confidence and reduce cognitive load?
- RQ2: How does human override impact trust and acceptance of predictions?

**Independent Variables:** explanation format (text vs. chart), override availability (on vs. off).
**Dependent Variables:** decision confidence, NASA-TLX, trust, SUS, calibration accuracy.
**Hypotheses:** H1 explanations increase confidence; H2 override reduces workload without lowering trust.

**Power analysis (example):** repeated-measures ANOVA, f=0.25, alpha=0.05, power=0.8 → ~64 participants.

**Recruitment & demographics:** students across disciplines with varied work hours and accessibility needs; collect GPA band, employment status, commute time.
**Ethics:** transparent disclosure of predictive model use and human override.


In [ ]:
# Example power analysis (two-sample t-test)
from statsmodels.stats.power import TTestIndPower
analysis = TTestIndPower()
sample = analysis.solve_power(effect_size=0.5, alpha=0.05, power=0.8)
sample


## 2) User Requirements

We use standardized instruments where possible (SUS, NASA-TLX) plus domain-specific Likert items.
Dummy survey responses are stored in `deliverables/data/`.

### Project 1: Survey Instrument
- Gaming habits, narrative/music priorities, accessibility needs, AI trust, co-creation interest.

### Project 2: Survey Instrument
- Planning confidence, workload characteristics, accessibility needs, and explanation preferences.


In [ ]:
import pandas as pd
p1 = pd.read_csv("deliverables/data/project1_user_requirements.csv")
p2 = pd.read_csv("deliverables/data/project2_user_requirements.csv")
p1.head(), p2.head()


## 3) Integrate Human Feedback

This step translates survey insights into UI and AI decisions.

### Project 1: Gemini-Orchestrated Multimodal Generation
- Story prompt conditioning from narrative preferences.
- Visual style selection from style distribution.
- Music generation constraints from music importance and accessibility needs.

### Project 2: AutoGluon + Human-Centered Explanations
- Feature selection based on perceived workload drivers.
- Explanation format aligned with user preferences.

#### LLM-based Code Generation (Gemini)
To use Gemini, set the environment variable `GEMINI_API_KEY` using the provided API key and run the code below.
If available in Google AI Studio, use **Gemini 3 Flash Lite** for low-cost generation; otherwise fall back to `gemini-1.5-flash`.


In [ ]:
import os
import google.generativeai as genai

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Set GEMINI_API_KEY before running this cell.")

genai.configure(api_key=api_key)
model_name = os.getenv("GEMINI_MODEL", "gemini-1.5-flash")
model = genai.GenerativeModel(model_name)

prompt = """You are a senior front-end engineer.
Generate HTML, CSS, and JavaScript for a static prototype that visualizes survey insights.
Use accessible design (WCAG AA), include: summary cards, preference charts (as inline SVG),
and a clear call to action to start co-creation.
Return three code blocks labeled HTML, CSS, JS."""

response = model.generate_content(prompt)
response.text


In [ ]:
import pandas as pd

survey = pd.read_csv("deliverables/data/project1_user_requirements.csv")
summary = survey.describe(include="all").to_string()

prompt = f"""You are generating a static web app from survey data.
Survey summary:
{summary}

Create accessible HTML/CSS/JS that visualizes the top user needs and preferences.
Return HTML, CSS, JS blocks only."""

# response = model.generate_content(prompt)
# response.text


#### Pre-built Deliverables
- `deliverables/project1_app/` and `deliverables/project2_app/` contain static prototypes.
- You can serve them with a simple local server: `python -m http.server` inside the folder.


## 4) Deploy Prototype

### Project 1 (Static Web App + Gemini Outputs)
1. Generate assets via Gemini (story, art style guidance, music descriptors).
2. Place outputs into a static web folder.
3. Host via `python -m http.server`.

### Project 2 (AutoGluon + Flask API)
- Train AutoGluon on `deliverables/data/project2_autogluon_training.csv`.
- Run a Flask app to expose `/predict` endpoint.
- Static UI calls the endpoint for predictions.


In [ ]:
from autogluon.tabular import TabularPredictor
import pandas as pd

train_df = pd.read_csv("deliverables/data/project2_autogluon_training.csv")
predictor = TabularPredictor(label="workload_score", path="deliverables/autogluon_model").fit(train_df)


In [ ]:
from flask import Flask, request, jsonify
import pandas as pd

app = Flask(__name__)
predictor = TabularPredictor.load("deliverables/autogluon_model")

@app.route("/predict", methods=["POST"])
def predict():
    payload = request.json
    df = pd.DataFrame([payload])
    pred = predictor.predict(df).iloc[0]
    return jsonify({"workload_score": float(pred)})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


## 5) User Evaluation

**Instruments:** SUS, NASA-TLX, trust scales, decision confidence, and open-ended feedback.
Dummy post-test data and interaction logs are in `deliverables/data/` and `deliverables/logs/`.

### Project 1
- Post-test survey: `project1_post_test.csv`
- Interaction logs: `interaction_logs.csv` (filtered by `project1`).

### Project 2
- Post-test survey: `project2_post_test.csv`
- Interaction logs: `interaction_logs.csv` (filtered by `project2`).


In [ ]:
post1 = pd.read_csv("deliverables/data/project1_post_test.csv")
post2 = pd.read_csv("deliverables/data/project2_post_test.csv")
logs = pd.read_csv("deliverables/logs/interaction_logs.csv")
post1.describe(include="all").T.head()


## 6) Analyses

### Quantitative Analyses
- Descriptive statistics, t-tests/ANOVA, effect sizes (Cohen’s d, partial η²), confidence intervals.
- Report reliability (Cronbach’s alpha) for scales.
- Mixed-method reporting: means, SDs, effect sizes, confidence intervals, and exact p-values.

### Qualitative Analyses
- LLM-assisted open coding with human verification.
- Manual coding using a shared codebook and double-coding.
- Inter-rater reliability: Cohen’s κ (nominal codes) or ICC (ordinal).

**Limitations of LLM-based coding:**
- Potential bias in code assignment and missing minority perspectives.
- Reduced transparency without explicit reasoning artifacts.
- Mitigations: blinded human audits, calibration rounds, and inter-rater checks.


In [ ]:
import numpy as np
from scipy import stats

# Example: compare SUS scores between projects
sus1 = post1["sus_score"]
sus2 = post2["sus_score"]
t_stat, p_val = stats.ttest_ind(sus1, sus2)
t_stat, p_val


In [ ]:
from sklearn.metrics import cohen_kappa_score

qual = pd.read_csv("deliverables/data/qualitative_annotations.csv")
# Example inter-rater reliability using primary codes
c1 = qual[qual["coder_id"] == "C1"]["code_primary"].values
c2 = qual[qual["coder_id"] == "C2"]["code_primary"].values
min_len = min(len(c1), len(c2))
kappa = cohen_kappa_score(c1[:min_len], c2[:min_len])
kappa


In [ ]:
# LLM-assisted qualitative analysis (outline)
import google.generativeai as genai

sample_quotes = qual["quote"].head(5).tolist()
prompt = """You are a qualitative researcher.
Assign open codes to each quote and propose 2-3 themes.
Return JSON with codes and themes."""

# response = model.generate_content(prompt + "
Quotes:" + "
".join(sample_quotes))
# response.text


## 7) Report (Example Research Paper)

**Title:** Human-Centered Orchestration of Multimodal AI in Creative and Planning Systems

### Abstract
We present two human-centered AI systems: a multimodal game co-creation platform and a workload planner using AutoML. Both are evaluated through mixed methods to assess trust, agency, and usability.

### Introduction
We position our work in human-AI co-creation, transparency, and inclusive design, framing the contributions for ACM CHI.

### Methods
- Participants: N=64 per project.
- Instruments: SUS, NASA-TLX, trust scale, open-ended feedback.
- Analysis: t-tests/ANOVA, effect sizes, and thematic analysis with inter-rater reliability.

### Results (Example Summaries)
- Project 1: Higher transparency increased trust and perceived agency.
- Project 2: Explanations increased decision confidence and reduced workload.

### Discussion
We discuss design implications for multimodal AI, value alignment, and participant agency.

### Limitations
- LLM-generated coding may introduce bias and require human audit.
- Dummy data limits ecological validity but supports classroom reproducibility.

### Conclusion
Human-centered methodologies enable rigorous, interpretable, and inclusive AI systems.


## References & Further Reading
- Amershi et al. (2019) Guidelines for Human-AI Interaction.
- Shneiderman (2020) Human-Centered AI.
- Brooke (1996) System Usability Scale (SUS).
- Hart & Staveland (1988) NASA-TLX.
- Braun & Clarke (2006) Thematic Analysis.
